# 01 Data Audit, Validation, and Cleaning

## Purpose

This notebook is the main Task 1 notebook for building a trustworthy shared transcript dataset.

It is designed to:
- load the raw transcript dataset through reusable `src/` logic
- inspect the working schema and key identifiers
- document and test three concrete extraction or cleaning issues
- apply a documented cleaning pipeline
- validate the cleaned output
- produce basic post-cleaning EDA for shared team use
- show the main outputs directly in the notebook and also save them to stable project paths

## Source Modules and Functions Used

- `src/data/load_transcripts.py`
  - `resolve_transcript_path`
  - `get_available_columns`
  - `load_raw_transcripts`
  - `compare_observed_to_expected_schema`
- `src/data/validate_transcripts.py`
  - `build_validation_report`
  - `summarize_missingness`
  - `summarize_date_coverage`
  - `summarize_firm_coverage`
  - `summarize_identifier_match_rate`
  - `summarize_text_length`
- `src/data/clean_transcripts.py`
  - `summarize_duplicate_event_groups`
  - `sample_duplicate_event_examples`
  - `summarize_preperiod_leakage`
  - `summarize_exact_zero_return_issue`
  - `apply_cleaning_pipeline`
- `src/data/build_panel.py`
  - `flag_usable_transcripts`
  - `summarize_usable_transcripts`
  - `filter_usable_transcripts`
  - `build_transcript_event_panel`
- `src/features/keyword_counts.py`
  - `build_keyword_feature_table`
- `src/utils/paths.py`
  - `ensure_project_dirs`
  - `transcript_raw_path`
  - `interim_data_path`
  - `processed_data_path`
  - `table_path`
  - `figure_path`

## Expected Inputs

- the raw transcript dataset in the canonical or legacy raw path
- a schema that includes transcript text, transcript identifiers, date fields, and finance linkage fields

## Outputs Written and Displayed

- audit and cleaning tables displayed in the notebook and saved to `outputs/tables/`
- EDA figures displayed in the notebook and saved to `outputs/figures/`
- cleaned transcript datasets saved to `data/interim/` and `data/processed/`
- a lightweight cleaned transcript event panel saved for later notebooks


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

# Make the repository root importable so `src/` works in Jupyter.
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT_CANDIDATES = [NOTEBOOK_DIR, NOTEBOOK_DIR.parent]
PROJECT_ROOT = next(
    (candidate for candidate in PROJECT_ROOT_CANDIDATES if (candidate / "src").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the repository root containing `src/`.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from src.config.schemas import REQUIRED_TRANSCRIPT_COLUMNS
from src.data.build_panel import (
    build_transcript_event_panel,
    filter_usable_transcripts,
    flag_usable_transcripts,
    summarize_usable_transcripts,
)
from src.data.clean_transcripts import (
    apply_cleaning_pipeline,
    sample_duplicate_event_examples,
    summarize_duplicate_event_groups,
    summarize_exact_zero_return_issue,
    summarize_preperiod_leakage,
)
from src.data.load_transcripts import (
    compare_observed_to_expected_schema,
    get_available_columns,
    load_raw_transcripts,
    resolve_transcript_path,
)
from src.data.validate_transcripts import (
    build_validation_report,
    summarize_date_coverage,
    summarize_firm_coverage,
    summarize_identifier_match_rate,
    summarize_missingness,
    summarize_text_length,
)
from src.features.keyword_counts import build_keyword_feature_table
from src.utils.paths import (
    ensure_project_dirs,
    figure_path,
    interim_data_path,
    processed_data_path,
    table_path,
    transcript_raw_path,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
sns.set_theme(style="whitegrid")
ensure_project_dirs()

load_nrows_env = os.environ.get("TASK1_LOAD_NROWS")
LOAD_NROWS = int(load_nrows_env) if load_nrows_env else None
RAW_PATH = transcript_raw_path()
MIN_CALL_DATE = "2010-01-01"
TEXT_COLUMN = "full_transcript_text"
DATE_COLUMN = "call_date"
CREATION_DATE_COLUMN = "transcriptcreationdate_utc"
FIRM_ID_COLUMN = "companyname"
TRANSCRIPT_ID_COLUMN = "transcriptid"
IDENTIFIER_COLUMNS = ["transcriptid", "companyid", "ticker", "permno", "gvkey", "ibes_ticker"]
DUPLICATE_GROUP_COLUMNS = ["ticker", "call_date"]
AUDIT_COLUMNS = None  # Load the full raw schema so the cleaned output preserves all columns.
KEYWORD_SETS = {
    "r_and_d": ["r&d", "research and development"],
    "dividends": ["dividend", "dividends", "buyback", "buybacks", "share repurchase"],
    "guidance": ["guidance", "outlook", "forecast"],
    "layoffs": ["layoff", "layoffs", "restructuring", "headcount reduction"],
    "acquisitions": ["acquisition", "acquisitions", "acquire", "merger", "m&a"],
}
OUTPUT_PREFIX = "01_data_audit"

print(f"Project root: {PROJECT_ROOT}")
print(f"Row limit: {'all rows' if LOAD_NROWS is None else LOAD_NROWS}")


## Raw Data Loading

This section uses the reusable loading logic in `src/data/load_transcripts.py` and restricts the working columns to those needed for Task 1.


In [ ]:
raw_path = resolve_transcript_path(RAW_PATH)
available_columns = get_available_columns(raw_path)
schema_comparison = compare_observed_to_expected_schema(raw_path)
transcripts = load_raw_transcripts(
    path=raw_path,
    columns=AUDIT_COLUMNS,
    nrows=LOAD_NROWS,
    parse_dates=True,
)

print(f"Loaded: {transcripts.shape[0]:,} rows, {transcripts.shape[1]} columns")
print(f"Date range: {transcripts[DATE_COLUMN].min()} to {transcripts[DATE_COLUMN].max()}")
print(f"Unique tickers: {transcripts['ticker'].nunique():,}")
print(f"Unique (ticker, call_date) pairs: {transcripts.groupby(DUPLICATE_GROUP_COLUMNS).ngroups:,}")
print(f"Raw transcript file: {raw_path}")
display(pd.DataFrame([schema_comparison]))
display(transcripts.head())


## Issue 1: Duplicate Transcript Revisions

This section mirrors the duplicate-revision review structure from your friend's notebook, but calls reusable logic in `src/data/clean_transcripts.py`.

Working assumption for Task 1:
- repeated `(ticker, call_date)` rows are likely multiple transcript revisions for the same call event
- the cleaning rule used later is to keep the longest transcript per `(ticker, call_date)`


In [ ]:
duplicate_issue = summarize_duplicate_event_groups(
    transcripts,
    group_cols=DUPLICATE_GROUP_COLUMNS,
)
duplicate_examples = sample_duplicate_event_examples(
    transcripts,
    group_cols=DUPLICATE_GROUP_COLUMNS,
    transcript_id_col=TRANSCRIPT_ID_COLUMN,
    text_col=TEXT_COLUMN,
    creation_col=CREATION_DATE_COLUMN,
    transcript_length_col="transcript_length",
)

dup_summary = duplicate_issue["summary"].iloc[0]
print(f"Total rows: {len(transcripts):,}")
print(f"Rows involved in duplicates: {int(dup_summary['rows_in_duplicate_groups']):,} ({dup_summary['rows_in_duplicate_groups_share']*100:.1f}%)")
print(f"Duplicate groups (>1 row per ticker+date): {int(dup_summary['duplicate_groups']):,}")
print(f"Max duplicates for one event: {int(dup_summary['max_group_size'])}")

print('\n' + '=' * 70)
print('EXAMPLES: Same call, multiple near-identical revisions')
print('=' * 70)

duplicate_groups = duplicate_issue['duplicate_groups']
example_groups = duplicate_groups.loc[duplicate_groups['row_count'] >= 3].copy()
if example_groups.empty:
    example_groups = duplicate_groups.copy()
example_groups = example_groups.sample(n=min(3, len(example_groups)), random_state=42) if not example_groups.empty else example_groups

for row in example_groups.itertuples(index=False):
    group_mask = (transcripts[DUPLICATE_GROUP_COLUMNS[0]] == getattr(row, DUPLICATE_GROUP_COLUMNS[0])) & (transcripts[DUPLICATE_GROUP_COLUMNS[1]] == getattr(row, DUPLICATE_GROUP_COLUMNS[1]))
    group = transcripts.loc[group_mask].copy()
    print(f"\n--- {getattr(row, DUPLICATE_GROUP_COLUMNS[0])} on {getattr(row, DUPLICATE_GROUP_COLUMNS[1])} ({int(row.row_count)} rows) ---")
    texts = group[TEXT_COLUMN].fillna('').astype(str).tolist()
    lengths = [len(text) for text in texts]
    for transcript_row in group.itertuples(index=False):
        txt = str(getattr(transcript_row, TEXT_COLUMN, ''))
        transcript_id = getattr(transcript_row, TRANSCRIPT_ID_COLUMN, None)
        created = getattr(transcript_row, CREATION_DATE_COLUMN, None)
        print(
            f"  transcriptid={transcript_id}  len={len(txt):,}  created={created}  first_80: {txt[:80]}"
        )
    if lengths:
        all_same = all(text == texts[0] for text in texts)
        pct_diff = ((max(lengths) - min(lengths)) / max(lengths) * 100) if max(lengths) else 0.0
        print(f"  -> Identical text? {all_same} | Length spread: {min(lengths):,}-{max(lengths):,} ({pct_diff:.1f}% diff)")

display(duplicate_issue["summary"])
display(duplicate_issue["duplicate_groups"].head(20))
display(duplicate_examples)


## Issue 2: Pre-2010 Data Leakage

This section tests whether rows before the intended sample window appear to have been included because of transcript creation or re-indexing dates rather than the actual call date.

The cleaning rule used later is:
- keep only `call_date >= 2010-01-01`


In [ ]:
preperiod_issue = summarize_preperiod_leakage(
    transcripts,
    call_date_col=DATE_COLUMN,
    creation_date_col=CREATION_DATE_COLUMN,
    min_call_date=MIN_CALL_DATE,
)

preperiod_summary = preperiod_issue["summary"].iloc[0]
print(f"Pre-2010 rows: {int(preperiod_summary['preperiod_rows']):,}")
print('\nDistribution by call year:')
call_year_counts = preperiod_issue['call_year_counts']
print(call_year_counts.set_index('call_year')['row_count'].to_string() if not call_year_counts.empty else 'None')
print('\nBut their transcriptcreationdate_utc is all post-2020:')
creation_year_counts = preperiod_issue['creation_year_counts']
print(creation_year_counts.set_index('creation_year')['row_count'].to_string() if not creation_year_counts.empty else 'None')
print(f"\nAll pre-2010 call_date rows have creation_date >= 2020? {bool(preperiod_summary['all_creation_dates_on_or_after_cutoff'])}")
print('\nSample rows (old calls re-indexed recently):')
sample_rows = preperiod_issue['sample_rows']
print(sample_rows.to_string(index=False) if not sample_rows.empty else 'None')

display(preperiod_issue["summary"])
display(call_year_counts)
display(creation_year_counts)
display(sample_rows)


## Issue 3: Exact-Zero Returns

This section checks for rows where the next-day open equals the prior close exactly, which may reflect stale or fallback pricing.

The cleaning rule used later is:
- drop rows where `close_price_call_day == open_price_next_day`


In [ ]:
zero_return_issue = summarize_exact_zero_return_issue(
    transcripts,
    return_col="close_to_open_return",
    close_col="close_price_call_day",
    open_col="open_price_next_day",
)

zero_ret = transcripts.loc[pd.to_numeric(transcripts['close_to_open_return'], errors='coerce').eq(0)].copy()
nonzero_ret = transcripts.loc[~pd.to_numeric(transcripts['close_to_open_return'], errors='coerce').eq(0)].copy()
zero_summary = zero_return_issue["summary"].iloc[0]
print(f"Zero-return rows: {int(zero_summary['zero_return_rows']):,}")
print(f"Of which close == open to the exact penny: {int(zero_summary['exact_close_open_matches']):,} / {int(zero_summary['zero_return_rows']):,} ({zero_summary['exact_match_share_within_zero_returns'] * 100:.1f}%)")
print(f"\nZero-return rows with close > $50: {int(zero_summary['high_price_zero_rows'])}")
print('\nExamples of high-price stocks with exactly zero return:')
zero_sample_rows = zero_return_issue['sample_rows']
print(zero_sample_rows.to_string(index=False) if not zero_sample_rows.empty else 'None')
print('\nPrice distribution comparison:')
print(f"{'':30s} {'Zero-return':>15s} {'Non-zero':>15s}")
print(f"{'Median price':30s} {pd.to_numeric(zero_ret['close_price_call_day'], errors='coerce').median():>15.2f} {pd.to_numeric(nonzero_ret['close_price_call_day'], errors='coerce').median():>15.2f}")
print(f"{'75th percentile':30s} {pd.to_numeric(zero_ret['close_price_call_day'], errors='coerce').quantile(0.75):>15.2f} {pd.to_numeric(nonzero_ret['close_price_call_day'], errors='coerce').quantile(0.75):>15.2f}")
print(f"{'% under $5':30s} {(pd.to_numeric(zero_ret['close_price_call_day'], errors='coerce') < 5).mean() * 100:>14.1f}% {(pd.to_numeric(nonzero_ret['close_price_call_day'], errors='coerce') < 5).mean() * 100:>14.1f}%")

display(zero_return_issue["summary"])
display(zero_sample_rows)


## Cleaning Pipeline

This section applies the three cleaning rules using `src/data/clean_transcripts.py`, then validates the cleaned dataset using the standard audit helpers in `src/data/validate_transcripts.py`.


In [ ]:
cleaned_transcripts, cleaning_log = apply_cleaning_pipeline(
    transcripts,
    call_date_col=DATE_COLUMN,
    min_call_date=MIN_CALL_DATE,
    dedup_group_cols=DUPLICATE_GROUP_COLUMNS,
    transcript_length_col="transcript_length",
    text_col=TEXT_COLUMN,
    close_col="close_price_call_day",
    open_col="open_price_next_day",
    remove_exact_zero_returns=True,
)

cleaned_validation_report = build_validation_report(
    cleaned_transcripts,
    required_columns=REQUIRED_TRANSCRIPT_COLUMNS,
    date_col=DATE_COLUMN,
    firm_id_col=FIRM_ID_COLUMN,
    text_col=TEXT_COLUMN,
    transcript_id_col=TRANSCRIPT_ID_COLUMN,
    identifier_columns=IDENTIFIER_COLUMNS,
)
cleaned_missingness_summary = summarize_missingness(cleaned_transcripts)
cleaned_identifier_match_rate = summarize_identifier_match_rate(cleaned_transcripts, IDENTIFIER_COLUMNS)
cleaned_date_coverage = summarize_date_coverage(cleaned_transcripts, DATE_COLUMN)
cleaned_firm_coverage = summarize_firm_coverage(cleaned_transcripts, FIRM_ID_COLUMN)
cleaned_text_length_report = summarize_text_length(cleaned_transcripts, TEXT_COLUMN, compute_from_text=False)

flagged_transcripts = flag_usable_transcripts(
    cleaned_transcripts,
    text_column=TEXT_COLUMN,
    date_column=DATE_COLUMN,
    transcript_id_column=TRANSCRIPT_ID_COLUMN,
)
usable_transcripts = filter_usable_transcripts(
    cleaned_transcripts,
    text_column=TEXT_COLUMN,
    date_column=DATE_COLUMN,
    transcript_id_column=TRANSCRIPT_ID_COLUMN,
)
usability_summary = summarize_usable_transcripts(flagged_transcripts)

print(f"Starting rows: {len(transcripts):,}")
step_labels = {
    'filter_preperiod_rows': 'Fix 1',
    'deduplicate_event_revisions': 'Fix 2',
    'remove_exact_zero_return_rows': 'Fix 3',
}
for row in cleaning_log.itertuples(index=False):
    label = step_labels.get(row.step, row.step)
    print(f"\n[{label}] {row.notes}: {row.rows_dropped:,} dropped -> {row.rows_after:,} remaining")
print(f"\n{'=' * 50}")
print(f"TOTAL: {len(transcripts):,} -> {len(cleaned_transcripts):,} rows ({len(cleaned_transcripts) / len(transcripts) * 100:.1f}% of original)")

print('\nVALIDATION CHECKS')
print('=' * 50)
dup_check = cleaned_transcripts.groupby(DUPLICATE_GROUP_COLUMNS).size()
print(f"[1] Max rows per (ticker, call_date): {dup_check.max()} (should be 1)")
min_date = pd.to_datetime(cleaned_transcripts[DATE_COLUMN], errors='coerce').min()
print(f"[2] Earliest call_date: {min_date.date()} (should be >= {MIN_CALL_DATE})")
exact_zeros = (pd.to_numeric(cleaned_transcripts['close_price_call_day'], errors='coerce') == pd.to_numeric(cleaned_transcripts['open_price_next_day'], errors='coerce')).sum()
print(f"[3] Exact-zero return rows: {int(exact_zeros)} (should be 0)")
print('\nFINAL DATASET')
print(f"  Rows: {len(cleaned_transcripts):,}")
print(f"  Columns: {len(cleaned_transcripts.columns)}")
print(f"  Date range: {cleaned_transcripts[DATE_COLUMN].min()} to {cleaned_transcripts[DATE_COLUMN].max()}")
print(f"  Unique tickers: {cleaned_transcripts['ticker'].nunique():,}")
print('\nTarget variable (close_to_open_return):')
ret = pd.to_numeric(cleaned_transcripts['close_to_open_return'], errors='coerce')
print(f"  Missing: {int(ret.isna().sum())}")
print(f"  Mean:   {ret.mean():.4f}")
print(f"  Median: {ret.median():.4f}")
print(f"  Std:    {ret.std():.4f}")
print(f"  Min:    {ret.min():.4f}")
print(f"  Max:    {ret.max():.4f}")

display(cleaning_log)
display(cleaned_validation_report["validation_summary"])
display(cleaned_missingness_summary.head(15))
display(cleaned_identifier_match_rate)
display(cleaned_date_coverage["summary"])
display(cleaned_firm_coverage["summary"])
display(cleaned_text_length_report["summary"])
display(usability_summary["overall"])
display(usability_summary["reasons"])


## Post-Cleaning EDA

This EDA is intentionally limited to the professor's immediate needs and uses the cleaned usable dataset rather than the raw file.


In [ ]:
date_by_year = cleaned_date_coverage["by_year"].copy()
top_firms = cleaned_firm_coverage["top_firms"].head(15).copy()
usable_text_metrics = cleaned_text_length_report["row_metrics"].loc[usable_transcripts.index].copy()
word_count_plot_col = "reported_word_count" if "reported_word_count" in usable_text_metrics.columns else usable_text_metrics.columns[0]

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=date_by_year, x="year", y="transcript_count", color="#4C78A8", ax=ax)
ax.set_title("Transcript Counts by Year After Cleaning")
ax.set_xlabel("Year")
ax.set_ylabel("Transcript Count")
fig.tight_layout()
fig.savefig(figure_path(f"{OUTPUT_PREFIX}_counts_by_year.png"), dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=top_firms, y=FIRM_ID_COLUMN, x="transcript_count", color="#F58518", ax=ax)
ax.set_title("Top Firms by Transcript Count After Cleaning")
ax.set_xlabel("Transcript Count")
ax.set_ylabel("Firm")
fig.tight_layout()
fig.savefig(figure_path(f"{OUTPUT_PREFIX}_top_firms.png"), dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
plot_word_count = usable_text_metrics[word_count_plot_col].dropna()
plot_word_count = plot_word_count.clip(upper=plot_word_count.quantile(0.99))
sns.histplot(plot_word_count, bins=40, color="#54A24B", ax=ax)
ax.set_title(f"{word_count_plot_col.replace('_', ' ').title()} Distribution After Cleaning")
ax.set_xlabel(word_count_plot_col.replace('_', ' ').title())
ax.set_ylabel("Number of Transcripts")
fig.tight_layout()
fig.savefig(figure_path(f"{OUTPUT_PREFIX}_word_count_distribution.png"), dpi=200, bbox_inches="tight")
plt.show()

keyword_features = build_keyword_feature_table(
    usable_transcripts,
    keyword_sets=KEYWORD_SETS,
    text_column=TEXT_COLUMN,
    id_column=TRANSCRIPT_ID_COLUMN,
)
keyword_summary_rows = []
for column in keyword_features.columns:
    if column == TRANSCRIPT_ID_COLUMN:
        continue
    series = keyword_features[column]
    keyword_summary_rows.append(
        {
            "keyword_theme": column.replace("_keyword_count", ""),
            "total_mentions": int(series.sum()),
            "transcripts_with_mentions": int((series > 0).sum()),
            "share_of_usable_transcripts": float((series > 0).mean()),
        }
    )
keyword_summary = pd.DataFrame(keyword_summary_rows).sort_values("transcripts_with_mentions", ascending=False)

display(date_by_year)
display(top_firms)
display(keyword_summary)


## Save Outputs

This section writes the cleaned shared dataset, audit tables, issue-proof tables, and post-cleaning EDA artifacts to stable project paths.


In [ ]:
tables_to_save = {
    "required_columns": cleaned_validation_report["required_columns"],
    "validation_summary": cleaned_validation_report["validation_summary"],
    "schema_summary": cleaned_validation_report["schema_summary"],
    "missingness_summary": cleaned_missingness_summary,
    "identifier_match_rate": cleaned_identifier_match_rate,
    "date_summary": cleaned_date_coverage["summary"],
    "date_by_year": date_by_year,
    "firm_summary": cleaned_firm_coverage["summary"],
    "top_firms": top_firms,
    "text_length_summary": cleaned_text_length_report["summary"],
    "usability_summary": usability_summary["overall"],
    "usability_reasons": usability_summary["reasons"],
    "keyword_summary": keyword_summary,
    "cleaning_log": cleaning_log,
    "duplicate_issue_summary": duplicate_issue["summary"],
    "duplicate_groups": duplicate_issue["duplicate_groups"],
    "duplicate_examples": duplicate_examples,
    "preperiod_issue_summary": preperiod_issue["summary"],
    "preperiod_call_year_counts": preperiod_issue["call_year_counts"],
    "preperiod_creation_year_counts": preperiod_issue["creation_year_counts"],
    "preperiod_sample_rows": preperiod_issue["sample_rows"],
    "zero_return_issue_summary": zero_return_issue["summary"],
    "zero_return_sample_rows": zero_return_issue["sample_rows"],
}

for name, table in tables_to_save.items():
    table.to_csv(table_path(f"{OUTPUT_PREFIX}_{name}.csv"), index=False)

flagged_output_path = interim_data_path(f"{OUTPUT_PREFIX}_flagged_cleaned_transcripts.parquet")
cleaned_output_path = processed_data_path(f"{OUTPUT_PREFIX}_cleaned_transcripts.parquet")
cleaned_csv_output_path = processed_data_path("CLEANED_earnings_calls_full_2010_onward_with_revenue.csv")
usable_output_path = processed_data_path(f"{OUTPUT_PREFIX}_usable_transcripts.parquet")
panel_output_path = interim_data_path(f"{OUTPUT_PREFIX}_transcript_event_panel.parquet")

flagged_transcripts.to_parquet(flagged_output_path, index=False)
cleaned_transcripts.to_parquet(cleaned_output_path, index=False)
cleaned_transcripts.to_csv(cleaned_csv_output_path, index=False)
usable_transcripts.to_parquet(usable_output_path, index=False)
build_transcript_event_panel(flagged_transcripts).to_parquet(panel_output_path, index=False)

save_manifest = pd.DataFrame(
    [
        {"artifact_type": "table", "artifact_name": name, "path": str(table_path(f"{OUTPUT_PREFIX}_{name}.csv"))}
        for name in tables_to_save
    ]
    + [
        {"artifact_type": "figure", "artifact_name": "counts_by_year", "path": str(figure_path(f"{OUTPUT_PREFIX}_counts_by_year.png"))},
        {"artifact_type": "figure", "artifact_name": "top_firms", "path": str(figure_path(f"{OUTPUT_PREFIX}_top_firms.png"))},
        {"artifact_type": "figure", "artifact_name": "word_count_distribution", "path": str(figure_path(f"{OUTPUT_PREFIX}_word_count_distribution.png"))},
        {"artifact_type": "dataset", "artifact_name": "flagged_cleaned_transcripts", "path": str(flagged_output_path)},
        {"artifact_type": "dataset", "artifact_name": "cleaned_transcripts", "path": str(cleaned_output_path)},
        {"artifact_type": "dataset", "artifact_name": "cleaned_transcripts_csv", "path": str(cleaned_csv_output_path)},
        {"artifact_type": "dataset", "artifact_name": "usable_transcripts", "path": str(usable_output_path)},
        {"artifact_type": "dataset", "artifact_name": "transcript_event_panel", "path": str(panel_output_path)},
    ]
)

print(f"Saved to: {cleaned_csv_output_path.name}")
print(f"Canonical save path: {cleaned_csv_output_path}")
print(f"Final size: {len(cleaned_transcripts):,} rows x {len(cleaned_transcripts.columns)} columns")
display(save_manifest)
display(cleaning_log)
display(cleaned_validation_report["validation_summary"])
display(keyword_summary)


## Final Notebook Summary

This notebook now combines three Task 1 responsibilities:
- audit the raw transcript dataset
- validate key schema and coverage properties
- apply a documented cleaning pipeline to create a cleaner shared transcript dataset

The notebook should leave the team with:
- issue-specific evidence tables for duplicate revisions, pre-period leakage, and exact-zero returns
- a cleaning log that records what each fix removed
- a cleaned transcript dataset for later notebooks
- post-cleaning descriptive outputs for basic EDA

Follow-up items still expected after this notebook:
- review the cleaning assumptions as a team before treating them as final canonical sample rules
- decide whether some rules belong in a finance-ready sample rather than the shared transcript master
- document final retained-versus-dropped counts in the data dictionary and methodology notes
